In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

1. 先點各section
2. 拿到各section 的子項目 (應該不用，新聞更新速度差)
3. 拿到個section 下的新聞

In [2]:
def setup_driver(driver_path):
    # Get the path for the ChromeDriver
    # driver_path = ChromeDriverManager().install()

    # Set up Chrome options or capabilities (if needed)
    chrome_options = webdriver.ChromeOptions()
    # chrome_options.add_argument("--headless")
    # chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-images")

    # Create a Chrome service with the driver path
    chrome_service = Service(driver_path)

    # Initialize the Chrome WebDriver with options and service
    driver = webdriver.Chrome(service=chrome_service, options=chrome_options)
    return driver

In [3]:
def scrape_news_link(driver, news_list, section_list, link_list, section_text):
    # button.click()
    # ===Get news url===
    html_content = driver.page_source
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')
    # res = requests.get('https://www.moneydj.com/kmdj/news/newsreallist.aspx?a=trends')
    # soup = BeautifulSoup(res.text, 'lxml')

    domain = 'https://www.moneydj.com/'
    for url in soup.select('.forumgrid tr a'):
        news_list.append('moneyDJ')
        section_list.append(section_text)
        link_list.append(domain + url.get('href'))
        # data_json['news'], data_json['section'], data_json['link'] = 'moneyDJ', section_text, domain + url.get('href')
        # print(domain + url.get('href'))
    return news_list, section_list, link_list

In [4]:
def click_section(driver, news_list, section_list, link_list, url = "https://www.moneydj.com/kmdj/news/newsreallist.aspx?a=trends"):
    # Navigate to the webpage
    driver.get(url)

    for i in range(21):
        section = 'ygtvlabelel' + str(i + 1)
        # Wait for the button to be clickable
        button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, section)))
        section_text = button.text
        print(section_text)
        button.click()
        # # Click the button
        # button.click()
        # Scrape the link of the news
        news_list, section_list, link_list = scrape_news_link(driver, news_list, section_list, link_list, section_text)
    return news_list, section_list, link_list



In [5]:
# Initialize the Chrome driver (outside the scraping function)
driver_path = ChromeDriverManager().install()
driver = setup_driver(driver_path)

# Get news link
news_list, section_list, link_list = [], [], []
news_list, section_list, link_list = click_section(driver, news_list, section_list, link_list)

# To dataframe
scraped_data = pd.DataFrame({
    'news': news_list,
    'section': section_list,
    'link': link_list
})
driver.quit()


市場動態
個股情報
產業分析
國家分析
債券
衍生性金融商品
期貨選擇權
個人理財
銀行理財
保險
房地產
黃金
稅務
基金投資
投資理論
交易實務
票券
投資典範
投資雜感
熱門產業
總體經濟


In [6]:
def get_content(url, title_list, time_list, article_list):
    res2 = requests.get(url)
    soup2 = BeautifulSoup(res2.text, 'lxml')
    title = soup2.select_one('h1').text
    time = soup2.find('span', {'id': 'MainContent_Contents_lbDate'}).text
    article = soup2.select_one('article').text
    title_list.append(title)
    time_list.append(time)
    article_list.append(article)
    print(title, '\n', time, '\n',article)
    return title_list, time_list, article_list

In [7]:
# Get text
title_list, time_list, article_list = [], [], []
for i in range(scraped_data.shape[0]):
    title_list, time_list, article_list = get_content(scraped_data['link'][i], title_list, time_list, article_list)
scraped_data['title'] = title_list
scraped_data['time'] = time_list
scraped_data['article'] = article_list


生合5/28參加櫃買舉辦之線上興櫃公司業績說明會 
 2024/05/24 19:13 
 

公開資訊觀測站重大訊息公告(1295)生合-本公司受邀參加財團法人中華民國證券櫃檯買賣中心舉辦之興櫃公司業績說明會符合條款第XX款：30事實發生日：113/05/281.召開法人說明會之日期：113/05/282.召開法人說明會之時間：14 時 00 分 3.召開法人說明會之地點：櫃買中心多功能資訊媒體區（台北市羅斯福路二段 100號11樓）4.法人說明會擇要訊息：本公司受邀參加財團法人中華民國證券櫃檯買賣中心舉辦之「興櫃公司業績說明會」，說明本公司之營運概況、財務及業務相關資訊。5.法人說明會簡報內容：內容檔案於當日會後公告於公開資訊觀測站6.公司網站是否有提供法人說明會內容：無7.其他應敘明事項：無。


《陸股》滬指跌0.88%失守3100點；恆指週K翻黑 
 2024/05/24 16:45 
 

MoneyDJ新聞 2024-05-24 16:45:41 記者 新聞中心 報導中國滬深兩市三大指數今(24)日集體續開低，早盤一度翻紅上揚，惟整體動能不足，隨後下挫翻黑；午後，三大指數跌幅加劇，滬指更跌破3100點。滬指終場收跌0.88%，連續二日下挫；本週累計跌2.07%，週K線翻黑。
上證指數今收3088.87點，下跌27.52點或0.88%；深證成指收於9424.58點，下跌117.06點或1.23%；創業板指收於1818.56點，下跌33.51點或1.81%；科創50收於727.21點，下跌15.19點或2.05%。以週線來看，上證指數本週跌2.07%、深證成指跌2.93%、創業板指跌2.49%，週K線均翻黑，科創50跌3.61%，週K線連3黑。
成交量方面，滬市今成交3,227億元(人民幣,下同)，深市成交4,412億元，兩市成交額7,639億元，較前一交易日量減約10%。
從盤面上來看，今日多數下跌，僅公用事業、農林漁牧上漲，房地產、電腦、電子、國防軍工、傳媒、社會服務等板塊跌幅居前。概念股中，電力、銅纜高速連接、豬肉、乳業等板塊漲幅居前，低空經濟、AI手機、房地產、記憶體晶片等板塊跌幅居前。
香港恆生指數今日開盤續跌0.67%，隨後震盪一路下挫，整日維持在低檔。盤面上，恆生科技指數跌2.48%，嗶哩嗶哩(9626.HK)跌逾11%，小米(1810.H

In [8]:
scraped_data.to_excel('新聞資料/moneyDJ0526.xlsx', index = False)